In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# Load municipality data
muni = gpd.read_file(r"C:\Users\lstojano\Desktop\teza\road_extraction_slovenia\data\raw\municipalities\municipalities_102109.gpkg")

# Basic info
print("Shape:", muni.shape)
print("\nColumns:", muni.columns.tolist())
print("\nCRS:", muni.crs)
print("\nFirst few rows:")
muni.head()

Shape: (212, 8)

Columns: ['FEATUREID', 'EID_OBCINA', 'SIFRA', 'NAZIV', 'OZNAKA_MES', 'DATUM_SYS', 'NAZIV_DJ', 'geometry']

CRS: PROJCS["ETRS_1989_Slovenia_TM",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9999],PARAMETER["false_easting",500000],PARAMETER["false_northing",-5000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102109"]]

First few rows:


,FEATUREID,EID_OBCINA,SIFRA,NAZIV,OZNAKA_MES,DATUM_SYS,NAZIV_DJ,geometry
0,OBCINE.110200000110277683,110200000110277683,49,Komen,0,2025-10-02,None,"MULTIPOLYGON (((402401.1 71863.73, 402441.17 7..."
1,OBCINE.110200000110275471,110200000110275471,126,Šoštanj,0,2025-10-02,None,"MULTIPOLYGON (((502701.01 135001.69, 502684.43..."
2,OBCINE.110200000214278371,110200000214278371,166,Križevci,0,2026-02-03,None,"MULTIPOLYGON (((586385.65 155234.43, 586373.72..."
3,OBCINE.110200000110281289,110200000110281289,146,Železniki,0,2025-10-02,None,"MULTIPOLYGON (((439801.09 123429.97, 439850.9 ..."
4,OBCINE.110200000110265167,110200000110265167,1,Ajdovščina,0,2025-12-05,None,"MULTIPOLYGON (((426457.04 78031.58, 426296.11 ..."


In [2]:
# Print all municipality names so we can match them exactly
all_names = sorted(muni['NAZIV'].tolist())
for name in all_names:
    print(name)

Ajdovščina
Ankaran
Apače
Beltinci
Benedikt
Bistrica ob Sotli
Bled
Bloke
Bohinj
Borovnica
Bovec
Braslovče
Brda
Brezovica
Brežice
Cankova
Celje
Cerklje na Gorenjskem
Cerknica
Cerkno
Cerkvenjak
Cirkulane
Destrnik
Divača
Dobje
Dobrepolje
Dobrna
Dobrova-Polhov Gradec
Dobrovnik
Dol pri Ljubljani
Dolenjske Toplice
Domžale
Dornava
Dravograd
Duplek
Gorenja vas-Poljane
Gorišnica
Gorje
Gornja Radgona
Gornji Grad
Gornji Petrovci
Grad
Grosuplje
Hajdina
Hodoš
Horjul
Hoče-Slivnica
Hrastnik
Hrpelje-Kozina
Idrija
Ig
Ilirska Bistrica
Ivančna Gorica
Izola
Jesenice
Jezersko
Juršinci
Kamnik
Kanal
Kidričevo
Kobarid
Kobilje
Komen
Komenda
Koper
Kostanjevica na Krki
Kostel
Kozje
Kočevje
Kranj
Kranjska Gora
Križevci
Krško
Kungota
Kuzma
Laško
Lenart
Lendava
Litija
Ljubljana
Ljubno
Ljutomer
Log-Dragomer
Logatec
Lovrenc na Pohorju
Loška dolina
Loški Potok
Lukovica
Luče
Majšperk
Makole
Maribor
Markovci
Medvode
Mengeš
Metlika
Mežica
Miklavž na Dravskem polju
Miren-Kostanjevica
Mirna
Mirna Peč
Mislinja
Mokronog-Trebe

In [3]:
import geopandas as gpd
import pandas as pd

# Load municipality data
muni = gpd.read_file(r"C:\Users\lstojano\Desktop\teza\road_extraction_slovenia\data\raw\municipalities\municipalities_102109.gpkg")

# Define our splits exactly as in config
splits = {
    'train': [
        'Ljubljana', 'Maribor', 'Koper', 'Celje', 'Ptuj',
        'Murska Sobota', 'Krško', 'Postojna', 'Kočevje',
        'Bovec', 'Ajdovščina', 'Slovenska Bistrica'
    ],
    'val': [
        'Kranj', 'Novo mesto', 'Tolmin'
    ],
    'test': [
        'Medvode', 'Domžale', 'Bohinj', 'Piran', 'Lendava'
    ]
}

# Build a flat list with split labels
rows = []
for split, names in splits.items():
    for name in names:
        rows.append({'NAZIV': name, 'split': split})

split_df = pd.DataFrame(rows)

# Filter municipality polygons to only our selected ones
all_selected = [name for names in splits.values() for name in names]
study = muni[muni['NAZIV'].isin(all_selected)].copy()

# Merge split labels in
study = study.merge(split_df, on='NAZIV', how='left')

# Keep and rename only the columns we need
study = study[['NAZIV', 'SIFRA', 'split', 'geometry']].copy()
study = study.rename(columns={'NAZIV': 'municipality', 'SIFRA': 'aoi_id'})

# Add landscape type column (we fill this in manually after)
study['landscape_type'] = ''
study['notes'] = ''

# Check result
print("Total selected municipalities:", len(study))
print("\nSplit counts:")
print(study['split'].value_counts())
print("\nAll selected:")
print(study[['municipality', 'split']].sort_values('split').to_string())

Total selected municipalities: 20

Split counts:
split
train    12
test      5
val       3
Name: count, dtype: int64

All selected:
          municipality  split
9                Piran   test
16             Domžale   test
15             Medvode   test
18             Lendava   test
19              Bohinj   test
6                 Ptuj  train
8        Murska Sobota  train
3              Kočevje  train
10            Postojna  train
11  Slovenska Bistrica  train
12               Celje  train
13           Ljubljana  train
14             Maribor  train
1                Koper  train
17               Krško  train
4                Bovec  train
0           Ajdovščina  train
5               Tolmin    val
7           Novo mesto    val
2                Kranj    val


In [4]:
import geopandas as gpd
import pandas as pd
import os

# Load municipality data
muni = gpd.read_file(r"C:\Users\lstojano\Desktop\teza\road_extraction_slovenia\data\raw\municipalities\municipalities_102109.gpkg")

# Define splits with Nova Gorica replacing Medvode in test
splits = {
    'train': [
        'Ljubljana', 'Maribor', 'Koper', 'Celje', 'Ptuj',
        'Murska Sobota', 'Krško', 'Postojna', 'Kočevje',
        'Bovec', 'Ajdovščina', 'Slovenska Bistrica'
    ],
    'val': [
        'Kranj', 'Novo mesto', 'Tolmin'
    ],
    'test': [
        'Nova Gorica', 'Domžale', 'Bohinj', 'Piran', 'Lendava'
    ]
}

# Landscape types
landscape_types = {
    'Ljubljana':           'urban',
    'Maribor':             'urban',
    'Koper':               'coastal_urban',
    'Celje':               'urban',
    'Ptuj':                'agricultural',
    'Murska Sobota':       'agricultural',
    'Krško':               'suburban',
    'Postojna':            'karst',
    'Kočevje':             'forest',
    'Bovec':               'alpine',
    'Ajdovščina':          'karst',
    'Slovenska Bistrica':  'agricultural',
    'Kranj':               'suburban_alpine',
    'Novo mesto':          'suburban',
    'Tolmin':              'alpine',
    'Nova Gorica':         'urban',
    'Domžale':             'suburban',
    'Bohinj':              'alpine',
    'Piran':               'coastal',
    'Lendava':             'agricultural'
}

# Build split dataframe
rows = []
for split, names in splits.items():
    for name in names:
        rows.append({'municipality': name, 'split': split})
split_df = pd.DataFrame(rows)

# Filter municipalities
all_selected = [n for names in splits.values() for n in names]
study = muni[muni['NAZIV'].isin(all_selected)].copy()

# Rename and merge
study = study.rename(columns={'NAZIV': 'municipality', 'SIFRA': 'aoi_id'})
study = study.merge(split_df, on='municipality', how='left')

# Add landscape type and notes
study['landscape_type'] = study['municipality'].map(landscape_types)
study['notes'] = ''

# Keep only needed columns
study = study[['aoi_id', 'municipality', 'split', 'landscape_type', 'notes', 'geometry']]

# Save to study_areas.gpkg
output_path = r"C:\Users\lstojano\Desktop\teza\road_extraction_slovenia\data\processed\study_areas.gpkg"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
study.to_file(output_path, driver='GPKG')

print("Saved successfully!")
print("\nFinal dataset:")
print(study[['municipality', 'split', 'landscape_type']].sort_values(['split', 'municipality']).to_string())
print("\nTotal:", len(study), "municipalities")

Saved successfully!

Final dataset:
          municipality  split   landscape_type
19              Bohinj   test           alpine
15             Domžale   test         suburban
18             Lendava   test     agricultural
16         Nova Gorica   test            urban
9                Piran   test          coastal
0           Ajdovščina  train            karst
4                Bovec  train           alpine
12               Celje  train            urban
1                Koper  train    coastal_urban
3              Kočevje  train           forest
17               Krško  train         suburban
13           Ljubljana  train            urban
14             Maribor  train            urban
8        Murska Sobota  train     agricultural
10            Postojna  train            karst
6                 Ptuj  train     agricultural
11  Slovenska Bistrica  train     agricultural
2                Kranj    val  suburban_alpine
7           Novo mesto    val         suburban
5               Tolmin  